<a href="https://colab.research.google.com/github/maital2711-bit/123/blob/TEST-BRANCH/ECG_Rhythm_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫀 ECG Rhythm Classifier from ECG Images
**Mai Siton, AI Development Course project**

Goal: look at a **picture of an ECG** and say which rhythm it is.

| Rhythm | Meaning |
|---|---|
| **SR** | Sinus rhythm |
| **AFIB** | Atrial fibrillation |
| **STACH** | Sinus tachycardia |
| **SBRAD** | Sinus bradycardia |

**Pipeline:**
```
**Pipeline:**
PTB-XL signals → pick 4 rhythms → split by patient → balance → filter (0.5–40 Hz)
→ draw as real-printout images (3×4 + lead II rhythm strip) → ResNet18 (transfer learning)
→ train last layer → fine-tune → evaluate → test on a real ECG printout
```

⚙️ Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

## 1. Setup
Install `wfdb` (reads ECG signal files) and import everything we need.

In [1]:
!pip install -q wfdb

import os, ast, time, shutil
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import kagglehub
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)   # should say: cuda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.3 MB/s eta 0:00:00
Device: cpu


## 2. The data: PTB-XL
- **21,837 ECGs** from ~18,000 real patients (Germany)
- Each ECG = **12 leads × 10 seconds**, stored as **signals (numbers)**, not pictures
- Each ECG has a row in `ptbxl_database.csv` with patient info + labels (`scp_codes`) written by cardiologists
- One ECG can have several labels (e.g. MI + sinus rhythm)

In [2]:
path = kagglehub.dataset_download("khyeh0719/ptb-xl-dataset")

# find the main folder (where the label table is)
for root, dirs, files in os.walk(path):
    if "ptbxl_database.csv" in files:
        ptbxl = root
        break
print("PTB-XL folder:", ptbxl)

df = pd.read_csv(os.path.join(ptbxl, "ptbxl_database.csv"))
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)   # text -> real dictionary

print("Number of ECGs:", len(df))
print("Patients:      ", df["patient_id"].nunique())
pd.set_option("display.max_colwidth", 100)
df[["ecg_id", "patient_id", "age", "sex", "scp_codes", "report"]].head()

Using Colab cache for faster access to the 'ptb-xl-dataset' dataset.
PTB-XL folder: /kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1
Number of ECGs: 21837
Patients:       18885


,ecg_id,patient_id,age,sex,scp_codes,report
0,1,15709.0,56.0,1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",sinusrhythmus periphere niederspannung
1,2,13243.0,19.0,0,"{'NORM': 80.0, 'SBRAD': 0.0}",sinusbradykardie sonst normales ekg
2,3,20372.0,37.0,1,"{'NORM': 100.0, 'SR': 0.0}",sinusrhythmus normales ekg
3,4,17014.0,24.0,0,"{'NORM': 100.0, 'SR': 0.0}",sinusrhythmus normales ekg
4,5,17448.0,19.0,1,"{'NORM': 100.0, 'SR': 0.0}",sinusrhythmus normales ekg


### Rhythm labels in PTB-XL
| Code | Meaning |
|---|---|
| SR | Sinus rhythm |
| AFIB | Atrial fibrillation |
| AFLT | Atrial flutter |
| STACH | Sinus tachycardia |
| SBRAD | Sinus bradycardia |
| SARRH | Sinus arrhythmia |
| SVTAC | Supraventricular tachycardia |
| PSVT | Paroxysmal SVT |
| SVARR | Supraventricular arrhythmia |
| BIGU / TRIGU | Bigeminy / Trigeminy |
| PACE | Paced rhythm |

Note: rhythm labels always show `0.0` as confidence. That does **not** mean "no". If the code is there, the rhythm is there.

In [3]:
rhythms = ["SR", "AFIB", "AFLT", "STACH", "SBRAD", "SARRH", "SVTAC", "PSVT", "BIGU", "TRIGU", "PACE", "SVARR"]

counts = Counter()
for codes in df["scp_codes"]:
    for r in rhythms:
        if r in codes:
            counts[r] += 1

for r, n in counts.most_common():
    print(f"{r:6s} -> {n}")

SR     -> 16782
AFIB   -> 1514
STACH  -> 826
SARRH  -> 772
SBRAD  -> 637
PACE   -> 296
SVARR  -> 157
BIGU   -> 82
AFLT   -> 73
SVTAC  -> 27
PSVT   -> 24
TRIGU  -> 20


## 3. Choose the rhythms
- We keep 4 rhythms with enough examples: **SR, AFIB, STACH, SBRAD**
- Rhythms with < 160 ECGs (flutter, SVT, bigeminy…) are too few to learn from
- **SARRH (sinus arrhythmia) is removed.** In earlier experiments the model could not separate it from SR (recall 17–33%). This is future work.
- Each ECG must have **exactly one** of our rhythms. ECGs with 0 or 2+ are skipped (too confusing for the model).

In [4]:
chosen = ["SR", "AFIB", "STACH", "SBRAD"]

def get_rhythm(codes):
    if "SARRH" in codes:                        # remove sinus arrhythmia completely
        return None
    found = [r for r in chosen if r in codes]   # which chosen rhythms this ECG has
    if len(found) == 1:                         # exactly one -> keep it
        return found[0]
    return None                                 # zero or more than one -> skip

df["rhythm"] = df["scp_codes"].apply(get_rhythm)   # new column (not in the original file)
data = df[df["rhythm"].notna()].copy()

print("ECGs kept:", len(data))
print(data["rhythm"].value_counts())

ECGs kept: 19754
rhythm
SR       16780
AFIB      1514
STACH      826
SBRAD      634
Name: count, dtype: int64


## 4. Split train / test by patient
- Some patients have **more than one ECG**
- If the same patient is in train **and** test, the model "recognizes the person" instead of the rhythm, which is data leakage
- So we split the **patients** 80/20 (`train_test_split` + `stratify`), and every ECG follows its patient

In [5]:
# 1. One row per patient, with that patient's rhythm (first ECG), used only for stratify
patients = data.groupby("patient_id")["rhythm"].first()

# 2. Split patients 80/20
train_pat, test_pat = train_test_split(
    patients.index, test_size=0.2, random_state=42, stratify=patients.values)

# 3. Each ECG goes where its patient went
train = data[data["patient_id"].isin(train_pat)]
test  = data[data["patient_id"].isin(test_pat)]

print("Train ECGs:", len(train), "| Test ECGs:", len(test))
print("Patients in both:", len(set(train["patient_id"]) & set(test["patient_id"])))   # must be 0

Train ECGs: 15822 | Test ECGs: 3932
Patients in both: 0


## 5. Balance the classes
- SR is ~85% of the data, so a lazy model could say "SR" every time and still look good
- We keep **1,500 SR** in train and **800 SR** in test, plus **all** the arrhythmias
- Bonus: fewer images means faster drawing and training

In [6]:
train_small = pd.concat([train[train["rhythm"] == "SR"].sample(1500, random_state=42),
                         train[train["rhythm"] != "SR"]])
test_small  = pd.concat([test[test["rhythm"] == "SR"].sample(800, random_state=42),
                         test[test["rhythm"] != "SR"]])

print("Train:\n", train_small["rhythm"].value_counts())
print("\nTest:\n",  test_small["rhythm"].value_counts())

Train:
 rhythm
SR       1500
AFIB     1215
STACH     665
SBRAD     509
Name: count, dtype: int64

Test:
 rhythm
SR       800
AFIB     299
STACH    161
SBRAD    125
Name: count, dtype: int64


## 6. Look at one ECG signal
- `wfdb.rdsamp` opens the signal file
- Shape **(1000, 12)** = 10 seconds × 100 samples/sec, 12 leads

In [7]:
row = train_small.iloc[0]
signal, info = wfdb.rdsamp(os.path.join(ptbxl, row["filename_lr"]))

print("Rhythm:", row["rhythm"])
print("Shape: ", signal.shape)
print("Leads: ", info["sig_name"])

Rhythm: SR
Shape:  (1000, 12)
Leads:  ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


## 7. Draw every ECG as a printout-style image
The model learns from **pictures** (so we can upload a real ECG later), but PTB-XL has only **numbers**, so we draw them once.

**Filter first (0.5–40 Hz):** removes baseline wander (breathing/movement) and muscle noise, like every ECG machine does. This improved AF detection.

**Layout = standard ECG printout:**
| Row | Content |
|---|---|
| 1 | I, aVR, V1, V4 (2.5 s each) |
| 2 | II, aVL, V2, V5 |
| 3 | III, aVF, V3, V6 |
| 4 | Lead II rhythm strip (10 s) |

**Why this layout:** real ECG printouts look like this, so the model can be tested on real printouts without tricks.
Earlier versions (12 tiny leads, then 3 big leads) worked on our drawings but didn't match real printouts.

- Image 896 × 512, pink grid every 0.2 s and 0.5 mV
- Drawn with PIL (fast, low RAM)
- Folder name = label (`ImageFolder` format)

⏱️ ~3 minutes. Files are deleted when Colab restarts, so just re-run.

In [ ]:
from scipy.signal import butter, filtfilt

def clean_ecg(signal, fs=100):
    b, a = butter(2, [0.5, 40], btype="band", fs=fs)   # keep only 0.5–40 Hz
    return filtfilt(b, a, signal, axis=0)

OUT = "/content/ecg_print"      # NEW folder for printout-style images
W, H = 896, 512                 # image size (like a page, 10 s wide)
ROW = H / 4                     # 4 rows: 3 rows of 12 leads + 1 rhythm strip = 128 px each
PINK = (255, 192, 203)

# standard printout layout: 3 rows × 4 columns, each column = 2.5 seconds
LAYOUT = [[0, 3, 6, 9],         # row 1: I,   aVR, V1, V4
          [1, 4, 7, 10],        # row 2: II,  aVL, V2, V5
          [2, 5, 8, 11]]        # row 3: III, aVF, V3, V6
RHYTHM_LEAD = 1                 # row 4: long lead II rhythm strip (10 s)

def draw_trace(d, sig, x0, x1, mid):
    xs = np.linspace(x0, x1, len(sig))
    ys = mid - np.clip(sig, -1.5, 1.5) * (ROW / 3)     # 1 mV ≈ 43 px
    d.line(list(zip(xs, ys)), fill="black", width=1)

def save_ecg_print(signal, path):
    img = Image.new("RGB", (W, H), "white")
    d = ImageDraw.Draw(img)
    for x in np.arange(0, W, W / 50):                  # grid: every 0.2 sec
        d.line([(x, 0), (x, H)], fill=PINK)
    for y in np.arange(0, H, ROW / 6):                 # grid: every 0.5 mV
        d.line([(0, y), (W, y)], fill=PINK)

    n = signal.shape[0]                                # 1000 samples = 10 s
    seg = n // 4                                       # 250 samples = 2.5 s
    colw = W / 4
    for r in range(3):                                 # rows 1–3: 12 leads, 2.5 s each
        mid = ROW * r + ROW / 2
        for c in range(4):
            lead = LAYOUT[r][c]
            draw_trace(d, signal[c * seg:(c + 1) * seg, lead], c * colw, (c + 1) * colw - 1, mid)
            if c > 0:                                  # small marker where the lead changes
                d.line([(c * colw, mid - 15), (c * colw, mid + 15)], fill="black")
    draw_trace(d, signal[:, RHYTHM_LEAD], 0, W - 1, ROW * 3 + ROW / 2)   # row 4: rhythm strip
    img.save(path)

for split, table in [("train", train_small), ("test", test_small)]:
    for _, row in tqdm(table.iterrows(), total=len(table), desc=split):
        folder = os.path.join(OUT, split, row["rhythm"])
        os.makedirs(folder, exist_ok=True)
        signal, _ = wfdb.rdsamp(os.path.join(ptbxl, row["filename_lr"]))
        signal = clean_ecg(signal)
        save_ecg_print(signal, os.path.join(folder, f"{row['ecg_id']}.png"))

print("DONE ✅")
afib = os.path.join(OUT, "train", "AFIB")
Image.open(os.path.join(afib, os.listdir(afib)[0]))

train:   0%|          | 0/3889 [00:00<?, ?it/s]

In [ ]:
# count images + show one per rhythm
for split in ["train", "test"]:
    print(f"\n{split}:")
    for cls in sorted(os.listdir(os.path.join(OUT, split))):
        print(f"  {cls:6s} -> {len(os.listdir(os.path.join(OUT, split, cls)))}")

classes = sorted(os.listdir(os.path.join(OUT, "train")))
fig, axes = plt.subplots(len(classes), 1, figsize=(12, 3 * len(classes)))
for ax, cls in zip(axes, classes):
    folder = os.path.join(OUT, "train", cls)
    ax.imshow(Image.open(os.path.join(folder, os.listdir(folder)[0])))
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Load the images (class style: `ImageFolder` + `DataLoader`)
- `ToTensor()`: picture → numbers between 0 and 1
- `Normalize(...)`: ImageNet mean/std, because ResNet was trained on ImageNet
- **No `Resize`**: we keep 896 × 512 so the details are not squeezed away (ResNet accepts any size)
- `DataLoader` hands the model 32 images at a time; `shuffle=True` mixes the order every epoch

In [ ]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(OUT, "train"), transform=tfm)
test_ds  = datasets.ImageFolder(os.path.join(OUT, "test"),  transform=tfm)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

class_names = train_ds.classes
print("Classes:", class_names)
print("Train:", len(train_ds), "| Test:", len(test_ds))

x, y = next(iter(train_loader))
print("One batch:", x.shape)     # [32, 3, 256, 768] = 32 images, 3 colors, 256 × 768 pixels

## 9. The model: ResNet18 + transfer learning
(from the class notebook *Transfer Learning & Fine-Tuning*)

```
[ ResNet18 body ]  ──→  [ last layer: fc ]
 already learned         replaced:
 lines, edges, shapes    1000 ImageNet classes → 4 rhythms
 🔒 FROZEN                ✏️ LEARNS
```
- **Freeze** means ResNet's knowledge is locked
- Only the new `fc` learns: 512 × 4 + 4 = **2,052** trainable parameters

In [ ]:
def read_frozen_model_from_ResNet18():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False          # freeze everything
    return model

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Frozen parameters:    {total - trainable:,}")

model = read_frozen_model_from_ResNet18()
model.fc = nn.Linear(model.fc.in_features, len(class_names))   # new last layer = 4 rhythms
model = model.to(device)
count_parameters(model)

## 10. Training function (class code)
Each **epoch**:
```
1. show 32 images        → model guesses
2. compare to the truth  → loss (how wrong)
3. loss.backward()       → compute gradients
4. optimizer.step()      → adjust the weights a little
5. repeat for all images, then check on test
```
- **train acc** = score on images it learns from
- **test acc** = score on new images, **the real score**

In [ ]:
training_log = []
criterion = nn.CrossEntropyLoss()

def train_model(model, train_loader, test_loader, optimizer, criterion, model_name, epochs=5):
    print(f"Training {model_name}...")
    for epoch in range(epochs):
        model.train()
        start_time = time.time()
        train_loss, train_correct, train_total = 0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        # evaluate on test
        model.eval()
        test_loss, test_correct, test_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, labels).item()
                _, preds = torch.max(outputs, 1)
                test_correct += (preds == labels).sum().item()
                test_total += labels.size(0)

        train_acc = train_correct / train_total
        test_acc = test_correct / test_total
        elapsed = time.time() - start_time
        print(f"Epoch {epoch}: train acc {train_acc:.2%} | test acc {test_acc:.2%} | {elapsed:.0f}s")

        training_log.append({"model": model_name, "epoch": epoch, "time_sec": elapsed,
                             "accuracy": train_acc, "loss": train_loss / len(train_loader),
                             "test_accuracy": test_acc, "test_loss": test_loss / len(test_loader)})

## 11. Stage 1: train only the last layer
- `filter(... requires_grad)` passes only the **unfrozen** parameters to the optimizer
- ⏱️ ~1 min per epoch

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
train_model(model, train_loader, test_loader, optimizer, criterion, "Final FC layer", epochs=3)

## 12. Stage 2: fine-tuning (unfreeze `layer4` + `fc`)
- ResNet learned on photos, so it doesn't know ECG shapes yet. Unlocking the last block lets it learn them.
- We **don't** reload the model, so we keep the `fc` trained in stage 1 (this was the "bug" question in the class notebook)
- Only **3 epochs**: in earlier runs, after ~2 epochs train acc went to 99% while test stayed flat, which is **overfitting** (memorizing)

In [ ]:
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True

count_parameters(model)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
train_model(model, train_loader, test_loader, optimizer, criterion, "Fine-Tuning", epochs=3)

## 13. Evaluate: confusion matrix + report
- **Recall** = of all real AFIB ECGs, how many did it catch?
- **Precision** = when it says AFIB, how often is it right?
- **Macro avg** = average over the 4 rhythms equally

In [ ]:
def report(m, loader, title="ECG Rhythm Confusion Matrix"):
    m.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            _, preds = torch.max(m(inputs.to(device)), 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
    plt.title(title)
    plt.show()
    print(classification_report(all_labels, all_preds, target_names=class_names))

report(model, test_loader)

In [ ]:
# training curves: train vs test accuracy
df_log = pd.DataFrame(training_log)
df_log["step"] = range(len(df_log))
plt.figure(figsize=(8, 4))
plt.plot(df_log["step"], df_log["accuracy"], marker="o", label="Train")
plt.plot(df_log["step"], df_log["test_accuracy"], marker="x", linestyle="--", label="Test")
plt.axvline(2.5, color="gray", linestyle=":", label="start fine-tuning")
plt.xlabel("Epoch (stage 1 → stage 2)")
plt.ylabel("Accuracy")
plt.title("Train vs Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 14. Save the trained model
Colab deletes everything on restart, so save the weights and download them.

In [ ]:
torch.save(model.state_dict(), "/content/ecg_resnet18_4rhythms.pth")
print("Saved ✅")

# from google.colab import files
# files.download("/content/ecg_resnet18_4rhythms.pth")   # uncomment to download to your computer

In [ ]:
torch.save(model_aug.state_dict(), "/content/ecg_resnet18_aug_3ep.pth")
print("Saved ✅")

In [ ]:
def compute_cam(image_pil, model, target_class=None):
    feats = []
    def hook_fn(module, inp, out):
        feats.append(out.detach())
    handle = model.layer4.register_forward_hook(hook_fn)   # grab layer4 output

    model.eval()
    x = tfm(image_pil.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
    handle.remove()

    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()   # confidence per rhythm
    pred_class = int(probs.argmax())
    used_class = pred_class if target_class is None else int(target_class)

    fmap = feats[0].squeeze(0)                               # [512, h, w]
    fc = model.fc[-1] if isinstance(model.fc, nn.Sequential) else model.fc   # our fc = Dropout + Linear
    w = fc.weight[used_class].detach()                       # [512]
    cam = torch.relu(torch.einsum("c,chw->hw", w, fmap))     # weighted sum of feature maps
    cam = cam - cam.min()
    cam = cam / cam.max().clamp_min(1e-8)

    cam_img = Image.fromarray((cam.cpu().numpy() * 255).astype(np.uint8)).resize(image_pil.size, Image.BILINEAR)
    return np.array(cam_img).astype(np.float32) / 255.0, pred_class, probs

def show_cam(image_pil, cam_01, true_name, pred_class, probs):
    img = np.array(image_pil.convert("RGB")).astype(np.float32) / 255.0
    heat = plt.cm.jet(cam_01)[..., :3]
    overlay = 0.5 * heat + 0.5 * img

    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    axes[0].imshow(img);     axes[0].set_title("Original");  axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("CAM overlay (red = where the model looked)"); axes[1].axis("off")
    conf = ", ".join(f"{c} {p:.0%}" for c, p in zip(class_names, probs))
    fig.suptitle(f"TRUE: {true_name}  |  PRED: {class_names[pred_class]}\n{conf}")
    plt.tight_layout()
    plt.show()

# one test ECG per rhythm
for cls in class_names:
    folder = os.path.join(OUT, "test", cls)
    img = Image.open(os.path.join(folder, sorted(os.listdir(folder))[0]))
    cam_01, pred, probs = compute_cam(img, model)
    show_cam(img, cam_01, cls, pred, probs)

## 16. Results history

| Run | Change | Classes | Accuracy | Macro F1 | AFIB recall | Notes |
|---|---|---|---|---|---|---|
| 0 | 12 small leads (224×224), FC 5 + fine-tune 5 epochs | 5 | 76% | 0.69 | 86% | Overfitting: train 99.5% vs test 76%. SARRH recall 33% |
| 1 | 3 big leads (II, V1, V5), 768×256, 3 + 3 epochs | 5 | 77% | 0.68 | 88% | SBRAD 74→84%, STACH 83→90%. SARRH recall 17% |
| 2 | SARRH removed | 4 | 87% | 0.83 | 88% | SBRAD precision only 0.62 |
| 3 | + augmentation + dropout | 4 | 88% | 0.85 | 81% | No overfitting (train 82% vs test 88%). SBRAD precision 0.78 |
| 3b | + 3 more epochs | 4 | 85% | 0.82 | 92% | More epochs = worse overall |
| 4 | Bandpass filter 0.5–40 Hz (no aug) | 4 | 88% | 0.84 | 91% | Best balance |
| 5 | Filter + aug + dropout | 4 | 83% | 0.80 | 83% | Aug did not help this time |
| 6 | Filter (re-run) | 4 | 85% | 0.82 | **93%** | Most sensitive: STACH 93%, SBRAD 89%, but SR 80%. CAM: all 4 examples correct, incl. fast AF |
| 7 | Printout layout (3×4 + lead II rhythm strip), filtered | 4 | _fill in_ | _fill in_ | _fill in_ | Matches real ECG printouts |

### What we did and why

**1. Chose the data: PTB-XL**
- 21,837 real ECGs, labeled by cardiologists
- Why: real 12-lead ECGs with rhythm labels (which arrhythmia), which other datasets didn't have

**2. Chose 4 rhythms: SR, AFIB, STACH, SBRAD**
- Why: enough examples to learn from
- Rare rhythms (flutter, SVT…) had too few ECGs

**3. Removed sinus arrhythmia (SARRH)**
- Why: the model couldn't tell it apart from SR (found only 17–33%), and it confused the other results

**4. Split by patient**
- Each patient is only in train or test
- Why: otherwise the model "recognizes the person", not the rhythm (cheating)

**5. Used fewer SR ECGs**
- Why: SR was ~85% of the data, so the model would just say "SR" all the time

**6. Drew the signals as pictures**
- Why: the goal is to upload an ECG photo, but PTB-XL has only numbers

**7. Made the leads bigger (3 leads instead of 12 tiny ones)**
- Leads II, V1, V5, each 4× taller and wider
- Why: in the small version P waves were ~1 pixel, so the model couldn't see them
- Result: slow/fast rhythms became much easier to recognize

**8. Cleaned the signal (filter 0.5–40 Hz)**
- Removed the wandering baseline (breathing/movement)
- Why: a fast AF ECG was called "sinus tach", and its baseline was moving a lot
- Result: better AF detection (up to 93%)

**9. Used a ready-trained model (ResNet18, transfer learning)**
- Why: it already knows how to "see" lines and shapes from 1.2M photos, so we only teach it ECGs

**10. Trained in 2 stages**
- First only the last layer, then also the last block (fine-tuning)
- Why: it learns ECG-specific shapes without memorizing

**11. Trained only 3 rounds (epochs) per stage**
- Why: with more rounds it started memorizing and got worse on new ECGs

**12. Tried augmentation + dropout**
- Why: to stop memorizing
- Result: not consistently better, so not used in the final model

**13. Checked where it looks (CAM)**
- Why: to see if it looks at the heartbeats like a cardiologist, not at random spots
- Result: it looks at the beats and the spacing between them

**14. Changed the images to a real printout layout**
- 3×4 grid (12 leads, 2.5 s each) + 10 s lead II rhythm strip
- Why: the 3-lead images didn't look like real ECG printouts, so the model couldn't be tested on a real one

### Important lesson
- The same code gives slightly different results each time (±3–5%), so one run alone isn't proof

### Final result
- 85–88% correct overall
- AF caught up to 93% of the time
- Trade-off: it sometimes calls a normal ECG an arrhythmia. For screening, that's better than missing AF.

now its time to

In [ ]:
path_real = kagglehub.dataset_download("kanishkarathore1604/ecg-image")

real = []
for root, dirs, fs in os.walk(path_real):
    for f in fs:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            real.append(os.path.join(root, f))
print("Real ECG images:", len(real))

img_real = Image.open(real[0]).convert("RGB")
print("File:", real[0])          # the folder name in the path = its label (e.g. Normal)
print("Size:", img_real.size)

plt.figure(figsize=(14, 8))
plt.imshow(img_real)
plt.show()

In [ ]:
torch.save(model.state_dict(), "/content/ecg_resnet18_printout.pth")
from google.colab import files
files.download("/content/ecg_resnet18_printout.pth")

In [ ]:
for i in [0, 1]:
    img_real = Image.open(real[i]).convert("RGB")
    img_in = img_real.crop((150, 285, 2130, 1510)).resize((W, H))
    cam_01, pred, probs = compute_cam(img_in, model)
    show_cam(img_in, cam_01, f"real printout {i}", pred, probs)

In [ ]:
print(real[0])
print(real[1])

In [ ]:
from google.colab import files

uploaded = files.upload()                  # click "Choose file" → Downloads → your image
fname = list(uploaded.keys())[0]

img_web = Image.open(fname).convert("RGB")
print("Size:", img_web.size)

plt.figure(figsize=(14, 6))
plt.imshow(img_web)                        # pixel numbers on the sides help you crop
plt.show()